# Notebook for measuring runtime of Hashing, bucketing and similarity value computation 

In [ ]:
import os
import sys
import numpy as np
import itertools
import pandas as pd

def find_project_root(target_folder="masteroppgave"):
    """Find the absolute path of a folder by searching upward."""
    currentdir = os.path.abspath("__file__")  # Get absolute script path
    while True:
        if os.path.basename(currentdir) == target_folder:
            return currentdir  # Found the target folder
        parentdir = os.path.dirname(currentdir)
        if parentdir == currentdir:  # Stop at filesystem root
            return None
        currentdir = parentdir  # Move one level up

# Example usage
project_root = find_project_root("masteroppgave")

if project_root:
    sys.path.append(project_root)
    print(f"Project root found: {project_root}")
else:
    raise RuntimeError("Could not find 'masteroppgave' directory")

from utils.helpers.measure_similarities import *


# Code for running several combinations of the runtime parameters

# Disk

In [9]:
MEASURE="disk_frechet_cy"
CITY="rome"
DATA_SIZE = [300]

#Parameters
DIAMETER_LIST = [0.2, 0.4, 0.6, 0.8, 1.0, 1.2, 1.4, 1.6, 1.8, 2.0, 2.2, 2.4, 2.6, 2.8, 3.0, 3.2, 3.4, 3.6, 3.8, 4.0]
DISKS_LIST = [10,20,30,40,50,60,70,80,90]
LAYERS_LIST = [1, 2, 3, 4, 5, 6]


#Strategies
BUCKETING_METHOD = "loose"
TRUE_TRAJECTORIES = False

#Logistics
PARALLEL_JOBS = 24
ITERATIONS = 1

In [10]:
if "disk_dtw_cy" or "disk_frechet_cy" in MEASURE:
    SCHEME = "disk"
elif "grid_dtw_cy" or "grid_frechet_cy" in MEASURE:
    SCHEME = "grid"
    
if "dtw" in MEASURE:
    measure = "dtw"
elif "frechet" in MEASURE:
    measure = "frechet"

#Filenames
if TRUE_TRAJECTORIES:
    folder = "true_trajectories"
    file_name = f"runtimes_bucketing({BUCKETING_METHOD})_(true_trajectories)_{CITY}_{measure}_{DATA_SIZE}_{SCHEME}.csv"
else:
    folder = "hashed_trajectories"
    file_name = f"runtimes_bucketing({BUCKETING_METHOD})_(hashed_trajectories)_{CITY}_{measure}_{DATA_SIZE}_{SCHEME}.csv"

output_path = f"../../../results_hashed/runtimes/bucketing/{CITY}/{folder}/{BUCKETING_METHOD}/{file_name}"
os.makedirs(os.path.dirname(output_path), exist_ok=True)

In [ ]:
print(f" Current param config: \n \tBUCKETING: YES \n\tBUCKETING_METHOD: {BUCKETING_METHOD}\n\tTRUE_TRAJECTORIES: {TRUE_TRAJECTORIES}\n\tCITY: {CITY}\n\tMEASURE: {MEASURE}\n\tDATA_SIZE: {DATA_SIZE}\n\tSCHEME: {SCHEME} \n\tPARALLEL_JOBS: {PARALLEL_JOBS} \n\tITERATIONS: {ITERATIONS} \n\n")

# Generate all combinations of parameters
param_combinations = list(itertools.product(DIAMETER_LIST, LAYERS_LIST, DISKS_LIST, DATA_SIZE))

first_write = True

# Iterate over each combination and run the function
for diameter, layers, disks, data_size in param_combinations:
    print(f" \n Running for Diameter: {diameter}, Layers: {layers}, Disks: {disks}, Data Size: {data_size}, True Trajectories: {TRUE_TRAJECTORIES}")


    if TRUE_TRAJECTORIES:
        df_result = compute_hashed_similarity_runtimes_with_bucketing_with_true_sim(
            measure=MEASURE,
            city=CITY,
            diameter=diameter,
            layers=layers,
            disks=disks,
            parallel_jobs=PARALLEL_JOBS,
            data_size=data_size,
            iterations=ITERATIONS,
            bucketing_method=BUCKETING_METHOD
        )
    else:
        df_result = compute_hashed_similarity_runtimes_with_bucketing(
            measure=MEASURE,
            city=CITY,
            diameter=diameter,
            layers=layers,
            disks=disks,
            parallel_jobs=PARALLEL_JOBS,
            data_size=data_size,
            iterations=ITERATIONS,
            bucketing_method=BUCKETING_METHOD
        )

    # Add parameters to result DataFrame
    df_result["City"] = CITY
    df_result["Measure"] = measure
    df_result["Diameter"] = diameter
    df_result["Layers"] = layers
    df_result["Disks"] = disks
    df_result["Size"] = data_size

    # Define the desired column order
    desired_order = ["City", "Measure", "Diameter", "Layers", "Disks","Size",
                "Average Similarity Computation Time (Seconds)", 
                "Average Hash Generation Time (Seconds)", 
                "Average Bucket Distribution Time (Seconds)", "Total time (Seconds)"]

    # Reorder columns
    df_result = df_result[desired_order]

    # Save the DataFrame to a CSV file
    df_result.to_csv(output_path, mode='a', header=first_write, index=False)
    first_write = False

# Grid

In [4]:
MEASURE="grid_dtw_cy"
CITY="rome"
DATA_SIZE = [300]

#Parameters
RESOLUTION_LIST = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3, 1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.0, 2.1, 2.2, 2.3, 2.4, 2.5, 2.6, 2.7, 2.8, 2.9, 3.0]
LAYERS_LIST = LAYERS_VALUES = [1, 2, 3, 4, 5, 6]

#Strategies
BUCKETING_METHOD = "original"
TRUE_TRAJECTORIES = False

#Logistics
PARALLEL_JOBS = 8
ITERATIONS = 1

In [5]:
if "disk_dtw_cy" or "disk_frechet_cy" in MEASURE:
    SCHEME = "disk"
if "grid_dtw_cy" or "grid_frechet_cy" in MEASURE:
    SCHEME = "grid"
    
if "dtw" in MEASURE:
    measure = "dtw"
elif "frechet" in MEASURE:
    measure = "frechet"

#Filenames
if TRUE_TRAJECTORIES:
    folder = "true_trajectories"
    file_name = f"runtimes_bucketing({BUCKETING_METHOD})_(true_trajectories)_{CITY}_{measure}_{DATA_SIZE}_{SCHEME}.csv"
else:
    folder = "hashed_trajectories"
    file_name = f"runtimes_bucketing({BUCKETING_METHOD})_(hashed_trajectories)_{CITY}_{measure}_{DATA_SIZE}_{SCHEME}.csv"

output_path = f"../../../results_hashed/runtimes/bucketing/{CITY}/{folder}/{BUCKETING_METHOD}/{file_name}"
os.makedirs(os.path.dirname(output_path), exist_ok=True)

In [ ]:
print(f" Current param config: \n \tBUCKETING: YES \n\tBUCKETING_METHOD: {BUCKETING_METHOD}\n\tTRUE_TRAJECTORIES: {TRUE_TRAJECTORIES}\n\tCITY: {CITY}\n\tMEASURE: {MEASURE}\n\tDATA_SIZE: {DATA_SIZE}\n\tSCHEME: {SCHEME} \n\tPARALLEL_JOBS: {PARALLEL_JOBS} \n\tITERATIONS: {ITERATIONS} \n\n")


# Generate all combinations of parameters
param_combinations = list(itertools.product(RESOLUTION_LIST, LAYERS_LIST, DATA_SIZE))

first_write = True

# Iterate over each combination and run the function
for resolution, layers, data_size in param_combinations:
    print(f"\nRunning for Resolution: {resolution}, Layers: {layers}, Data Size: {data_size}, True Trajectories: {TRUE_TRAJECTORIES}")

    if TRUE_TRAJECTORIES:
        df_result = compute_hashed_similarity_runtimes_with_bucketing_with_true_sim(
            measure=MEASURE,
            city=CITY,
            res=resolution,  # Grid uses resolution instead of diameter
            layers=layers,
            parallel_jobs=PARALLEL_JOBS,
            data_size=data_size,
            iterations=ITERATIONS,
            bucketing_method=BUCKETING_METHOD
        )
    else:
        df_result = compute_hashed_similarity_runtimes_with_bucketing(
            measure=MEASURE,
            city=CITY,
            res=resolution,  # Grid uses resolution instead of diameter
            layers=layers,
            parallel_jobs=PARALLEL_JOBS,
            data_size=data_size,
            iterations=ITERATIONS,
            bucketing_method=BUCKETING_METHOD
        )

    # Add parameters to result DataFrame
    df_result["City"] = CITY
    df_result["Measure"] = measure
    df_result["Resolution"] = resolution
    df_result["Layers"] = layers
    df_result["Size"] = data_size

    # Define the desired column order
    desired_order = [
        "City", "Measure", "Resolution", "Layers", "Size",
        "Average Similarity Computation Time (Seconds)", 
        "Average Hash Generation Time (Seconds)", 
        "Average Bucket Distribution Time (Seconds)",
        "Total time (Seconds)"]

    # Reorder columns
    df_result = df_result[desired_order]

    # Save the DataFrame to a CSV file
    df_result.to_csv(output_path, mode='a', header=first_write, index=False)
    first_write = False
